In [10]:
# Cell 1 — Imports and paths
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_labels_1 = {
    "0": "T cells (resting)", "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells", "3": "Activated T cells",
    "4": "Macrophages", "5": "Monocytes/DC"
}

print("Ready")

Ready


In [7]:
# Cell 2 — Load raw counts and add cell type labels
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase1_v2_rawcounts.h5ad")

labels1 = sc.read_h5ad(
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad"
).obs[["leiden_0.8"]].copy()

labels1["cell_type"] = labels1["leiden_0.8"].map(cluster_labels_1)
adata1.obs["cell_type"] = labels1["cell_type"].reindex(adata1.obs_names).values

gc.collect()

print(adata1)
print(adata1.obs["cell_type"].value_counts())
print("Tissue types:", adata1.obs["tissue"].unique().tolist())
print("Max value:", adata1.X.max())

AnnData object with n_obs × n_vars = 44662 × 14800
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'cell_type'
    var: 'n_cells'
    uns: 'log1p'
cell_type
T cells (resting)         11734
Macrophages                8691
NK/Cytotoxic T cells       8676
Activated T cells          7141
T cells (naive/memory)     4341
Monocytes/DC               4079
Name: count, dtype: int64
Tissue types: ['TUMOR', 'NORMAL', 'LYMPHNODE', 'BLOOD']
Max value: 8.170489


In [8]:
# Cell 3 — Pseudobulk aggregation function
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        
        counts_list.append(X_sample.sum(axis=0))
        
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample,
            condition_col
        ].iloc[0]
        
        meta_list.append({
            sample_col: sample,
            condition_col: condition
        })
    
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    
    return counts_df, meta_df

print("Function defined")

Function defined


In [9]:
# Cell 4 — Run pseudobulk DE: T cells (resting) TUMOR vs BLOOD
counts_df, meta_df = pseudobulk_aggregate(
    adata1,
    cell_type="T cells (resting)",
    sample_col="patient",
    condition_col="tissue"
)

# Filter to TUMOR vs BLOOD only
mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
counts_df = counts_df[mask]
meta_df = meta_df[mask]

print("\nFiltered metadata:")
print(meta_df)
print("\nCounts matrix shape:", counts_df.shape)
print("Sample counts (first 5 genes):")
print(counts_df.iloc[:, :5])


T cells (resting): 11734 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}

Filtered metadata:
        tissue
patient       
BC5      TUMOR
BC6      TUMOR
BC4      BLOOD
BC8      TUMOR
BC1      BLOOD

Counts matrix shape: (5, 14800)
Sample counts (first 5 genes):
     A1BG  A2M  A4GALT  AAAS  AACS
BC5     9    2       2     8     4
BC6    43   45       0    12    10
BC4   509   45       2   305   154
BC8    72   55       0    39     8
BC1   293   15       0    63    47


In [11]:
# Cell 5 — Run PyDESeq2
inference = DefaultInference()

dds = DeseqDataSet(
    counts=counts_df,
    metadata=meta_df,
    design="~tissue",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

stat_res = DeseqStats(
    dds,
    contrast=["tissue", "TUMOR", "BLOOD"],
    inference=inference
)
stat_res.summary()

results_df = stat_res.results_df
print("\nDE results shape:", results_df.shape)
print(results_df.head(10))

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 7.83 seconds.

Fitting dispersion trend curve...
... done in 1.21 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 7.12 seconds.

Fitting LFCs...
... done in 5.27 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.07 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    100.505627       -0.953306  0.545830 -1.746525  0.080720  0.453021
A2M      38.319755        2.742585  0.799644  3.429760  0.000604  0.015392
A4GALT    1.166872        2.949114  3.613279  0.816188  0.414393       NaN
AAAS     39.956549       -0.392882  0.624035 -0.629584  0.528967  0.910009
AACS     19.915768       -1.041107  0.762207 -1.365911  0.171967  0.644533
...            ...             ...       ...       ...       ...       ...
ZXDC     39.553216       -0.438461  0.598639 -0.732430  0.463906  0.883077
ZYG11B   52.172902       -0.550829  0.519488 -1.060330  0.288995  0.768913
ZYX     364.060762        0.001300  0.357036  0.003642  0.997094  0.999625
ZZEF1    89.007546       -0.211750  0.402319 -0.526323  0.598664  0.925099
ZZZ3     35.950273       -0.653884  0.581977 -1.123556  0.261201  0.741917

[14800 rows x 6 columns]

DE results sh

In [12]:
# Cell 6 — Filter significant results and save
sig_df = results_df[
    (results_df["padj"] < 0.05) &
    (abs(results_df["log2FoldChange"]) > 0.5)
].copy()

sig_df = sig_df.sort_values("padj")

print(f"Significant DEGs: {len(sig_df)}")
print("\nTop upregulated in TUMOR:")
print(sig_df[sig_df["log2FoldChange"] > 0].head(10)[["log2FoldChange", "padj"]])
print("\nTop downregulated in TUMOR:")
print(sig_df[sig_df["log2FoldChange"] < 0].head(10)[["log2FoldChange", "padj"]])

results_df.to_csv(
    RESULTS_DIR / "GSE114725_DE_Tcells_resting_tumor_vs_blood.csv"
)
sig_df.to_csv(
    RESULTS_DIR / "GSE114725_DE_Tcells_resting_tumor_vs_blood_significant.csv"
)
print("\nSaved")

Significant DEGs: 601

Top upregulated in TUMOR:
          log2FoldChange          padj
HLA-DRA         2.370183  1.301097e-24
ATF3            5.026530  3.558711e-24
LYZ             4.212372  5.649935e-22
CCL3            3.146662  4.076705e-19
DUSP4           3.847187  1.191198e-17
SGK1            3.209662  5.102780e-16
CD163           6.082647  1.108901e-15
HLA-DPA1        2.084260  1.108901e-15
DUSP2           1.988918  7.242165e-15
KLF4            3.992785  2.182498e-14

Top downregulated in TUMOR:
                log2FoldChange          padj
TIPIN                -3.769220  1.867508e-21
NOSIP                -2.562704  6.041420e-14
CTD-2192J16.22       -2.049773  1.308714e-13
RP11-255M2.3         -4.497167  4.607259e-10
LINC00861            -2.440483  4.157993e-09
TCF7                 -1.499511  2.343567e-08
NACA2                -2.010141  2.485961e-08
GLTSCR2              -1.272002  3.404337e-08
EEF1B2               -1.212180  1.053353e-07
RPL29                -1.330355  5.292750e-0

In [14]:
# Cell 7 — Loop DE across all cell types
cell_types_de = [
    "T cells (naive/memory)",
    "NK/Cytotoxic T cells",
    "Activated T cells",
    "Macrophages"
]

all_results = {}

for ct in cell_types_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)
    
    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata1,
            cell_type=ct,
            sample_col="patient",
            condition_col="tissue"
        )
        
        # Filter to TUMOR vs BLOOD
        mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
        counts_df = counts_df[mask]
        meta_df = meta_df[mask]
        
        # Need at least 2 samples per condition
        condition_counts = meta_df["tissue"].value_counts()
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples: {condition_counts.to_dict()}")
            continue
        
        # Run PyDESeq2
        inference = DefaultInference()
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~tissue",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()
        
        stat_res = DeseqStats(
            dds,
            contrast=["tissue", "TUMOR", "BLOOD"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df
        
        # Filter significant
        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")
        
        print(f"  Significant DEGs: {len(sig_df)}")
        
        # Save
        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood.csv"
        )
        sig_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood_significant.csv"
        )
        
        all_results[ct] = {"full": results_df, "sig": sig_df}
        
    except Exception as e:
        print(f"  Failed: {e}")

print("\nAll DE analyses complete")


Running DE for: T cells (naive/memory)

T cells (naive/memory): 4341 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}


Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 7.22 seconds.

Fitting dispersion trend curve...
... done in 1.14 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 6.44 seconds.

Fitting LFCs...
... done in 5.38 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.11 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue    padj
A1BG     28.044609       -1.172274  0.681098 -1.721154  0.085223     NaN
A2M       6.646336        2.093911  1.388028  1.508551  0.131414     NaN
A4GALT    0.199973       -0.367400  4.675395 -0.078582  0.937365     NaN
AAAS      8.681757       -0.195319  0.995869 -0.196129  0.844509     NaN
AACS      6.589275       -1.854132  1.303401 -1.422534  0.154871     NaN
...            ...             ...       ...       ...       ...     ...
ZXDC      9.743484       -0.948449  1.232691 -0.769413  0.441648     NaN
ZYG11B   15.762889        0.046730  0.866388  0.053937  0.956986     NaN
ZYX     102.646024       -0.254668  0.530821 -0.479763  0.631396  0.9106
ZZEF1    27.237556       -0.675252  0.603449 -1.118989  0.263145     NaN
ZZZ3     12.914374       -0.786872  0.967945 -0.812930  0.416258     NaN

[14800 rows x 6 columns]
  Significant DEGs: 202

Running DE fo

...

Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     86.408201       -0.750495  0.523422 -1.433824  0.151622  0.745231
A2M      79.002431        0.185112  0.670598  0.276041  0.782517  0.992030
A4GALT    2.647931       -3.604479  2.561484 -1.407184  0.159373       NaN
AAAS     37.271207        0.189574  0.596551  0.317784  0.750649  0.991373
AACS     19.703979        0.172865  0.802294  0.215463  0.829406  0.993304
...            ...             ...       ...       ...       ...       ...
ZXDC     34.663375       -0.300173  0.693118 -0.433077  0.664959  0.988068
ZYG11B   49.556252       -0.085941  0.599526 -0.143349  0.886015  0.993304
ZYX     400.118863        0.141006  0.323827  0.435435  0.663247  0.988068
ZZEF1    92.445476       -0.217478  0.399879 -0.543860  0.586538  0.980608
ZZZ3     37.009228       -0.999472  0.603132 -1.657137  0.097492  0.643106

[14800 rows x 6 columns]
  Significant 

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 12.29 seconds.

Fitting dispersion trend curve...
... done in 1.77 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 13.48 seconds.

Fitting LFCs...
... done in 8.44 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.94 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    109.034313       -1.001316  0.505816 -1.979604  0.047748  0.284723
A2M     304.658203        1.464778  1.401143  1.045416  0.295831  0.694602
A4GALT    3.809885        0.629523  2.284968  0.275506  0.782927       NaN
AAAS     50.421978       -0.217259  0.582478 -0.372991  0.709155  0.917662
AACS     26.083255        0.442139  0.813638  0.543410  0.586848  0.872952
...            ...             ...       ...       ...       ...       ...
ZXDC     51.213505       -0.241679  0.645750 -0.374262  0.708210  0.917412
ZYG11B   85.119188        0.293456  0.569757  0.515055  0.606515  0.881999
ZYX     616.538593        0.536644  0.368485  1.456354  0.145295  0.513422
ZZEF1   122.941934        0.207719  0.470969  0.441047  0.659179  0.900910
ZZZ3     58.664243       -0.542610  0.574551 -0.944407  0.344962  0.731527

[14800 rows x 6 columns]
  Significant 

In [15]:
# Cell 8 — Volcano plots for all cell types
def volcano_plot(results_csv, title, save_path, lfc_thresh=0.5, pval_thresh=0.05):
    de_df = pd.read_csv(results_csv, index_col=0)
    de_df = de_df.dropna(subset=["padj", "log2FoldChange"])
    de_df["-log10_pval"] = -np.log10(de_df["padj"].clip(lower=1e-300))
    
    de_df["colour"] = "grey"
    de_df.loc[
        (de_df["log2FoldChange"] > lfc_thresh) & (de_df["padj"] < pval_thresh),
        "colour"
    ] = "red"
    de_df.loc[
        (de_df["log2FoldChange"] < -lfc_thresh) & (de_df["padj"] < pval_thresh),
        "colour"
    ] = "blue"
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for colour, group in de_df.groupby("colour"):
        ax.scatter(
            group["log2FoldChange"],
            group["-log10_pval"],
            c=colour,
            alpha=0.5,
            s=10,
            label=colour
        )
    
    ax.axvline(x=lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axvline(x=-lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(y=-np.log10(pval_thresh), color="black", linestyle="--", linewidth=0.8)
    
    # Label top genes
    top_up = de_df[de_df["colour"] == "red"].nlargest(5, "-log10_pval")
    top_down = de_df[de_df["colour"] == "blue"].nlargest(5, "-log10_pval")
    
    for gene, row in pd.concat([top_up, top_down]).iterrows():
        ax.annotate(
            gene,
            (row["log2FoldChange"], row["-log10_pval"]),
            fontsize=7,
            ha="center",
            xytext=(0, 5),
            textcoords="offset points"
        )
    
    n_up = (de_df["colour"] == "red").sum()
    n_down = (de_df["colour"] == "blue").sum()
    
    ax.text(0.98, 0.98, f"Up: {n_up}\nDown: {n_down}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9)
    
    ax.set_xlabel("Log2 Fold Change (Tumour vs Blood)")
    ax.set_ylabel("-log10 Adjusted P-value")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {save_path.name}")

# Plot for all cell types
cell_types_plot = {
    "T_cells__resting_": "T cells (resting)",
    "T_cells__naive_memory_": "T cells (naive/memory)",
    "NK_Cytotoxic_T_cells": "NK/Cytotoxic T cells",
    "Activated_T_cells": "Activated T cells",
    "Macrophages": "Macrophages"
}

for ct_file, ct_label in cell_types_plot.items():
    csv_path = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood.csv"
    if csv_path.exists():
        volcano_plot(
            csv_path,
            title=f"{ct_label} — Tumour vs Blood (GSE114725)",
            save_path=FIGURE_DIR / f"GSE114725_volcano_{ct_file}.png"
        )
    else:
        print(f"File not found: {csv_path.name}")

File not found: GSE114725_DE_T_cells__resting__tumor_vs_blood.csv
File not found: GSE114725_DE_T_cells__naive_memory__tumor_vs_blood.csv
Saved: GSE114725_volcano_NK_Cytotoxic_T_cells.png
Saved: GSE114725_volcano_Activated_T_cells.png
Saved: GSE114725_volcano_Macrophages.png


In [16]:
# Cell 9 — Pathway enrichment on DE results
import gseapy as gp

cell_types_enrich = {
    "T_cells__resting_": "T cells (resting)",
    "T_cells__naive_memory_": "T cells (naive/memory)",
    "NK_Cytotoxic_T_cells": "NK/Cytotoxic T cells",
    "Activated_T_cells": "Activated T cells",
    "Macrophages": "Macrophages"
}

for ct_file, ct_label in cell_types_enrich.items():
    print(f"\n{'='*50}")
    print(f"Pathway enrichment: {ct_label}")
    
    # Load full results for background gene list
    full_csv = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood.csv"
    sig_csv = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood_significant.csv"
    
    if not full_csv.exists():
        print(f"  File not found: {full_csv.name}")
        continue
    
    full_df = pd.read_csv(full_csv, index_col=0).dropna(subset=["padj"])
    sig_df = pd.read_csv(sig_csv, index_col=0)
    
    # Background = all genes tested in this cell type
    background = full_df.index.tolist()
    
    # Upregulated in tumour
    up_genes = sig_df[sig_df["log2FoldChange"] > 0].index.tolist()
    # Downregulated in tumour
    down_genes = sig_df[sig_df["log2FoldChange"] < 0].index.tolist()
    
    print(f"  Up: {len(up_genes)}, Down: {len(down_genes)}, Background: {len(background)}")
    
    for direction, gene_list in [("up", up_genes), ("down", down_genes)]:
        if len(gene_list) < 10:
            print(f"  Skipping {direction} - too few genes ({len(gene_list)})")
            continue
        
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
                background=background,
                outdir=None,
                verbose=False
            )
            
            sig_paths = enr.results[enr.results["Adjusted P-value"] < 0.05].copy()
            print(f"  {direction}: {len(sig_paths)} significant pathways")
            
            if len(sig_paths) > 0:
                print(sig_paths[["Gene_set", "Term", "Adjusted P-value"]].head(5).to_string())
            
            enr.results.to_csv(
                RESULTS_DIR / f"GSE114725_pathways_{ct_file}_{direction}.csv",
                index=False
            )
            
        except Exception as e:
            print(f"  {direction} enrichment failed: {e}")

print("\nPathway enrichment complete")


Pathway enrichment: T cells (resting)
  File not found: GSE114725_DE_T_cells__resting__tumor_vs_blood.csv

Pathway enrichment: T cells (naive/memory)
  File not found: GSE114725_DE_T_cells__naive_memory__tumor_vs_blood.csv

Pathway enrichment: NK/Cytotoxic T cells
  Up: 196, Down: 109, Background: 9953
  up: 93 significant pathways
               Gene_set                           Term  Adjusted P-value
0  MSigDB_Hallmark_2020  TNF-alpha Signaling via NF-kB      5.407267e-50
1  MSigDB_Hallmark_2020      Interferon Gamma Response      5.659096e-21
2  MSigDB_Hallmark_2020          Inflammatory Response      1.513241e-16
3  MSigDB_Hallmark_2020            Allograft Rejection      6.846337e-12
4  MSigDB_Hallmark_2020           IL-2/STAT5 Signaling      1.525446e-11
  down: 3 significant pathways
                Gene_set                 Term  Adjusted P-value
0   MSigDB_Hallmark_2020       Myc Targets V1      1.927493e-05
15       KEGG_2021_Human             Ribosome      3.780338e-86
16  

In [17]:
import os
# Check what files actually exist
de_files = list(RESULTS_DIR.glob("GSE114725_DE_T_cells*"))
for f in de_files:
    print(f.name)

GSE114725_DE_T_cells_(naive_memory)_tumor_vs_blood.csv
GSE114725_DE_T_cells_(naive_memory)_tumor_vs_blood_significant.csv


In [18]:
de_files = list(RESULTS_DIR.glob("GSE114725_DE_*"))
for f in sorted(de_files):
    print(f.name)

GSE114725_DE_Activated_T_cells_tumor_vs_blood.csv
GSE114725_DE_Activated_T_cells_tumor_vs_blood_significant.csv
GSE114725_DE_Macrophages_tumor_vs_blood.csv
GSE114725_DE_Macrophages_tumor_vs_blood_significant.csv
GSE114725_DE_NK_Cytotoxic_T_cells_tumor_vs_blood.csv
GSE114725_DE_NK_Cytotoxic_T_cells_tumor_vs_blood_significant.csv
GSE114725_DE_T_cells_(naive_memory)_tumor_vs_blood.csv
GSE114725_DE_T_cells_(naive_memory)_tumor_vs_blood_significant.csv
GSE114725_DE_Tcells_resting_tumor_vs_blood.csv
GSE114725_DE_Tcells_resting_tumor_vs_blood_significant.csv


In [19]:
# Fix filenames for missing cell types
missing_types = {
    "Tcells_resting": "T cells (resting)",
    "T_cells_(naive_memory)": "T cells (naive/memory)"
}

for ct_file, ct_label in missing_types.items():
    print(f"\n{'='*50}")
    print(f"Pathway enrichment: {ct_label}")
    
    full_csv = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood.csv"
    sig_csv = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood_significant.csv"
    
    full_df = pd.read_csv(full_csv, index_col=0).dropna(subset=["padj"])
    sig_df = pd.read_csv(sig_csv, index_col=0)
    
    background = full_df.index.tolist()
    up_genes = sig_df[sig_df["log2FoldChange"] > 0].index.tolist()
    down_genes = sig_df[sig_df["log2FoldChange"] < 0].index.tolist()
    
    print(f"  Up: {len(up_genes)}, Down: {len(down_genes)}, Background: {len(background)}")
    
    for direction, gene_list in [("up", up_genes), ("down", down_genes)]:
        if len(gene_list) < 10:
            print(f"  Skipping {direction} - too few genes ({len(gene_list)})")
            continue
        
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
                background=background,
                outdir=None,
                verbose=False
            )
            
            sig_paths = enr.results[enr.results["Adjusted P-value"] < 0.05].copy()
            print(f"  {direction}: {len(sig_paths)} significant pathways")
            
            if len(sig_paths) > 0:
                print(sig_paths[["Gene_set", "Term", "Adjusted P-value"]].head(5).to_string())
            
            enr.results.to_csv(
                RESULTS_DIR / f"GSE114725_pathways_{ct_file}_{direction}.csv",
                index=False
            )
            
        except Exception as e:
            print(f"  {direction} enrichment failed: {e}")

print("\nDone")


Pathway enrichment: T cells (resting)
  Up: 385, Down: 216, Background: 10981
  up: 115 significant pathways
               Gene_set                           Term  Adjusted P-value
0  MSigDB_Hallmark_2020  TNF-alpha Signaling via NF-kB      1.371867e-57
1  MSigDB_Hallmark_2020            Allograft Rejection      2.316889e-21
2  MSigDB_Hallmark_2020          Inflammatory Response      2.280190e-20
3  MSigDB_Hallmark_2020      Interferon Gamma Response      3.712767e-19
4  MSigDB_Hallmark_2020                      Apoptosis      2.930629e-17
  down: 3 significant pathways
                Gene_set                 Term  Adjusted P-value
0   MSigDB_Hallmark_2020       Myc Targets V1      2.110488e-07
26       KEGG_2021_Human             Ribosome     4.377337e-101
27       KEGG_2021_Human  Coronavirus disease      6.855373e-90

Pathway enrichment: T cells (naive/memory)
  Up: 92, Down: 110, Background: 2597
  up: 22 significant pathways
               Gene_set                              

Restart kernel before running cells below. GSE176078 and GSE114725 cannot be loaded in the same session due to RAM constraints.

In [1]:
# Cell 11 — Imports for GSE176078 analysis
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gseapy as gp
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Monocytes/DC", "16": "Cycling myeloid", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial", "21": "Epithelial",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

print("Ready")

Ready


In [2]:
# Cell 12 — Load GSE176078 raw counts and add labels
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase1_v2_rawcounts.h5ad")

labels2 = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad"
).obs[["leiden_0.6", "subtype", "orig.ident"]].copy()

labels2["cell_type"] = labels2["leiden_0.6"].map(cluster_labels_2)

adata2.obs["cell_type"] = labels2["cell_type"].reindex(adata2.obs_names).values
adata2.obs["subtype"] = labels2["subtype"].reindex(adata2.obs_names).values
adata2.obs["orig.ident"] = labels2["orig.ident"].reindex(adata2.obs_names).values

gc.collect()

print(adata2)
print(adata2.obs["cell_type"].value_counts())
print("Subtypes:", adata2.obs["subtype"].unique().tolist())
print("Max value:", adata2.X.max())

AnnData object with n_obs × n_vars = 91425 × 27343
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'cell_type'
    var: 'n_cells'
    uns: 'log1p'
cell_type
Luminal epithelial      22894
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
Endothelial cells        7040
CAFs                     6453
T cells                  5989
PVL                      5051
Cycling epithelial       3010
B cells                  2791
Plasma cells             2583
NK cells                 2438
Cycling cells            1159
Basal epithelial         1074
Epithelial                905
pDC                       315
Monocytes/DC               22
Cycling myeloid            19
Name: count, dtype: int64
Subtypes: ['HER2+', 'TNBC', 'ER+']
Max value: 9.017809


In [3]:
# Cell 13 — Pseudobulk aggregation function (same as GSE114725)
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        counts_list.append(X_sample.sum(axis=0))
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample, condition_col
        ].iloc[0]
        meta_list.append({sample_col: sample, condition_col: condition})
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    return counts_df, meta_df

print("Function defined")

Function defined


In [4]:
# Cell 14 — Loop DE across immune cell types (subtype comparisons)
immune_types_de = [
    "T cells", "CD8 T cells", "Macrophages", "NK cells",
    "B cells", "Naive/memory T cells"
]

all_results2 = {}

for ct in immune_types_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata2,
            cell_type=ct,
            sample_col="orig.ident",
            condition_col="subtype"
        )

        # Check enough samples per condition
        condition_counts = meta_df["subtype"].value_counts()
        print(f"  Samples per subtype: {condition_counts.to_dict()}")
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples")
            continue

        # Run PyDESeq2 — all subtypes
        inference = DefaultInference()
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~subtype",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        ct_clean = ct.replace("/", "_").replace(" ", "_")

        # Three pairwise comparisons
        comparisons = [
            ("TNBC", "ER+"),
            ("TNBC", "HER2+"),
            ("HER2+", "ER+")
        ]

        for test, ref in comparisons:
            try:
                stat_res = DeseqStats(
                    dds,
                    contrast=["subtype", test, ref],
                    inference=inference
                )
                stat_res.summary()
                results_df = stat_res.results_df

                sig_df = results_df[
                    (results_df["padj"] < 0.05) &
                    (abs(results_df["log2FoldChange"]) > 0.5)
                ].copy().sort_values("padj")

                print(f"  {test} vs {ref}: {len(sig_df)} significant DEGs")

                results_df.to_csv(
                    RESULTS_DIR / f"GSE176078_DE_{ct_clean}_{test}_vs_{ref}.csv"
                )
                sig_df.to_csv(
                    RESULTS_DIR / f"GSE176078_DE_{ct_clean}_{test}_vs_{ref}_significant.csv"
                )

                all_results2[f"{ct}_{test}_vs_{ref}"] = {
                    "full": results_df, "sig": sig_df
                }

            except Exception as e:
                print(f"  {test} vs {ref} failed: {e}")

    except Exception as e:
        print(f"  Failed: {e}")

print("\nAll GSE176078 DE complete")


Running DE for: T cells

T cells: 5989 cells
  Pseudobulk matrix: (25, 27343)
  Conditions: {'TNBC': 10, 'ER+': 10, 'HER2+': 5}
  Samples per subtype: {'TNBC': 10, 'ER+': 10, 'HER2+': 5}


Fitting size factors...
... done in 0.10 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 9.84 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 2.78 seconds.

Fitting MAP dispersions...
... done in 23.29 seconds.

Fitting LFCs...
... done in 27.58 seconds.

Calculating cook's distance...
... done in 0.20 seconds.

Replacing 109 outlier genes.

Fitting dispersions...
... done in 0.41 seconds.

Fitting MAP dispersions...
... done in 0.39 seconds.

Fitting LFCs...
... done in 0.42 seconds.

Running Wald tests...
... done in 6.13 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.630236        1.291336  3.533270  0.365479  0.714754   
FO538757.3     0.000000             NaN       NaN       NaN       NaN   
FO538757.2     8.872396       -0.644556  0.546521 -1.179379  0.238247   
AP006222.2     0.932280        0.248082  0.910723  0.272401  0.785314   
RP4-669L17.10  0.019939        0.512717  2.672063  0.191881  0.847836   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.999994  
FO538757.3   

...

Log2 fold change & Wald test p-value: subtype TNBC vs ER+
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7    0.064901       -0.700471  1.654521 -0.423368  0.672027   
FO538757.3      0.032494       -0.917118  2.974986 -0.308276  0.757872   
FO538757.2     12.834083       -0.168148  0.482337 -0.348610  0.727382   
AP006222.2      2.494910       -0.073856  0.872011 -0.084696  0.932503   
RP4-669L17.10   0.166502       -0.191400  1.670610 -0.114569  0.908787   
...                  ...             ...       ...       ...       ...   
MTNR1B          0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2    0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8    0.000000             NaN       NaN       NaN       NaN   
LINC01570       0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4    0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.999092  
F

Running Wald tests...
... done in 5.30 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs HER2+
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7    0.064901        2.736993  2.259185  1.211496  0.225705   
FO538757.3      0.032494        2.463926  3.721364  0.662103  0.507905   
FO538757.2     12.834083        0.057028  0.552310  0.103253  0.917762   
AP006222.2      2.494910       -0.211091  0.972360 -0.217092  0.828137   
RP4-669L17.10   0.166502        2.096277  2.005924  1.045043  0.296003   
...                  ...             ...       ...       ...       ...   
MTNR1B          0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2    0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8    0.000000             NaN       NaN       NaN       NaN   
LINC01570       0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4    0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.973759  

Running Wald tests...
... done in 5.03 seconds.



Log2 fold change & Wald test p-value: subtype HER2+ vs ER+
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7    0.064901       -3.437464  2.231170 -1.540655  0.123401   
FO538757.3      0.032494       -3.381044  3.589572 -0.941907  0.346240   
FO538757.2     12.834083       -0.225175  0.538572 -0.418097  0.675876   
AP006222.2      2.494910        0.137235  0.956864  0.143422  0.885957   
RP4-669L17.10   0.166502       -2.287677  1.992210 -1.148311  0.250840   
...                  ...             ...       ...       ...       ...   
MTNR1B          0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2    0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8    0.000000             NaN       NaN       NaN       NaN   
LINC01570       0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4    0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.968650  


GSE176078 — ER+ vs TNBC only (excluding HER2+ due to low sample size, n=5)

In [1]:
# Cell 1 — Imports
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Monocytes/DC", "16": "Cycling myeloid", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial", "21": "Epithelial",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

print("Ready")

Ready


In [2]:
# Cell 2 — Load GSE176078 raw counts
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase1_v2_rawcounts.h5ad")

labels2 = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad"
).obs[["leiden_0.6", "subtype", "orig.ident"]].copy()

labels2["cell_type"] = labels2["leiden_0.6"].map(cluster_labels_2)

adata2.obs["cell_type"] = labels2["cell_type"].reindex(adata2.obs_names).values
adata2.obs["subtype"] = labels2["subtype"].reindex(adata2.obs_names).values
adata2.obs["orig.ident"] = labels2["orig.ident"].reindex(adata2.obs_names).values

# Filter to ER+ and TNBC only — exclude HER2+
adata2_ertnbc = adata2[adata2.obs["subtype"].isin(["ER+", "TNBC"])].copy()
del adata2
gc.collect()

print(f"ER+ and TNBC cells: {adata2_ertnbc.n_obs}")
print(adata2_ertnbc.obs["subtype"].value_counts())
print(adata2_ertnbc.obs["cell_type"].value_counts())

ER+ and TNBC cells: 73061
subtype
TNBC    39509
ER+     33552
Name: count, dtype: int64
cell_type
Luminal epithelial      20723
Macrophages              7408
Naive/memory T cells     7025
CD8 T cells              6484
Endothelial cells        6070
CAFs                     5112
PVL                      4136
T cells                  4008
Cycling epithelial       2730
Plasma cells             2463
B cells                  2228
NK cells                 1599
Cycling cells            1033
Epithelial                890
Basal epithelial          864
pDC                       247
Monocytes/DC               22
Cycling myeloid            19
Name: count, dtype: int64


In [4]:
# Cell 4 — Loop DE across immune cell types (ER+ vs TNBC)
immune_types_de = [
    "T cells", "CD8 T cells", "Macrophages", "NK cells",
    "B cells", "Naive/memory T cells"
]

all_results_ertnbc = {}

for ct in immune_types_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata2_ertnbc,
            cell_type=ct,
            sample_col="orig.ident",
            condition_col="subtype"
        )

        condition_counts = meta_df["subtype"].value_counts()
        print(f"  Samples per subtype: {condition_counts.to_dict()}")
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples")
            continue

        inference = DefaultInference()
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~subtype",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["subtype", "TNBC", "ER+"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df

        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")

        print(f"  Significant DEGs (TNBC vs ER+): {len(sig_df)}")
        if len(sig_df) > 0:
            print("\n  Top upregulated in TNBC:")
            print(sig_df[sig_df["log2FoldChange"] > 0].head(5)[["log2FoldChange", "padj"]].to_string())
            print("\n  Top downregulated in TNBC:")
            print(sig_df[sig_df["log2FoldChange"] < 0].head(5)[["log2FoldChange", "padj"]].to_string())

        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(RESULTS_DIR / f"GSE176078_DE_{ct_clean}_TNBC_vs_ERplus.csv")
        sig_df.to_csv(RESULTS_DIR / f"GSE176078_DE_{ct_clean}_TNBC_vs_ERplus_significant.csv")

        all_results_ertnbc[ct] = {"full": results_df, "sig": sig_df}

    except Exception as e:
        print(f"  Failed: {e}")

print("\nAll ER+ vs TNBC DE complete")


Running DE for: T cells

T cells: 4008 cells
  Pseudobulk matrix: (20, 27343)
  Conditions: {'TNBC': 10, 'ER+': 10}
  Samples per subtype: {'TNBC': 10, 'ER+': 10}


Fitting size factors...
... done in 0.10 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 11.72 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 1.95 seconds.

Fitting MAP dispersions...
... done in 21.09 seconds.

Fitting LFCs...
... done in 21.06 seconds.

Calculating cook's distance...
... done in 0.14 seconds.

Replacing 89 outlier genes.

Fitting dispersions...
... done in 0.47 seconds.

Fitting MAP dispersions...
... done in 0.39 seconds.

Fitting LFCs...
... done in 0.44 seconds.

Running Wald tests...
... done in 7.95 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.567704        1.251105  3.213017  0.389386  0.696990   
FO538757.3     0.000000             NaN       NaN       NaN       NaN   
FO538757.2     6.585138       -0.671928  0.602944 -1.114413  0.265102   
AP006222.2     0.372613        0.297691  0.972227  0.306195  0.759456   
RP4-669L17.10  0.018127        0.470901  2.160901  0.217919  0.827493   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.999979  
FO538757.3   

...

Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.027778       -2.551427  2.471270 -1.032436  0.301868   
FO538757.3     0.022593       -1.395301  2.479699 -0.562690  0.573646   
FO538757.2     2.160396        0.086597  0.949909  0.091164  0.927363   
AP006222.2     0.426921        0.191732  1.066073  0.179849  0.857271   
RP4-669L17.10  0.021918       -1.491941  2.497910 -0.597276  0.550323   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.940447  
FO538757.3   

Fitting size factors...
... done in 0.07 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 11.46 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 1.73 seconds.

Fitting MAP dispersions...
... done in 15.75 seconds.

Fitting LFCs...
... done in 15.85 seconds.

Calculating cook's distance...
... done in 0.07 seconds.

Replacing 147 outlier genes.

Fitting dispersions...
... done in 0.48 seconds.

Fitting MAP dispersions...
... done in 0.48 seconds.

Fitting LFCs...
... done in 0.51 seconds.

Running Wald tests...
... done in 6.49 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue     padj
RP11-34P13.7   0.057903       -0.761338  1.346229 -0.565534  0.571711  0.99976
FO538757.3     0.029007       -0.947996  2.296532 -0.412795  0.679757  0.99976
FO538757.2     9.242977       -0.176969  0.525966 -0.336465  0.736520  0.99976
AP006222.2     1.623876       -0.062328  0.903580 -0.068979  0.945007  0.99976
RP4-669L17.10  0.125807       -0.224993  1.494271 -0.150570  0.880315  0.99976
...                 ...             ...       ...       ...       ...      ...
MTNR1B         0.000000             NaN       NaN       NaN       NaN      NaN
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN      NaN
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN      NaN
LINC01570      0.000000             NaN       NaN       NaN       NaN      NaN
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN    

T cell Subclusters DE

In [5]:
# Cell 2 — Load T cell subclustered object and raw counts
adata1_tcells = sc.read_h5ad(PROCESSED_DIR / "GSE114725_tcells_subclustered.h5ad")

adata1_raw = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase1_v2_rawcounts.h5ad")

# Subset raw counts to T cell barcodes only
tcell_barcodes = adata1_tcells.obs_names
adata1_tcells_raw = adata1_raw[tcell_barcodes].copy()
del adata1_raw
gc.collect()

# Transfer labels
adata1_tcells_raw.obs["tcell_subtype"] = adata1_tcells.obs["tcell_subtype"].values
adata1_tcells_raw.obs["tissue"] = adata1_tcells.obs["tissue"].values
adata1_tcells_raw.obs["patient"] = adata1_tcells.obs["patient"].values

print(adata1_tcells_raw)
print(adata1_tcells_raw.obs["tcell_subtype"].value_counts())
print("Tissues:", adata1_tcells_raw.obs["tissue"].unique().tolist())
print("Max value:", adata1_tcells_raw.X.max())

AnnData object with n_obs × n_vars = 31892 × 14800
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'tcell_subtype'
    var: 'n_cells'
    uns: 'log1p'
tcell_subtype
Naive/Memory T cells    8835
NK/Cytotoxic T cells    6529
Activated T cells       6200
Resting T cells         5336
B cell contamination    4992
Name: count, dtype: int64
Tissues: ['TUMOR', 'NORMAL', 'LYMPHNODE', 'BLOOD']
Max value: 8.170489


In [6]:
# Cell 3 — Pseudobulk aggregation function
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="tcell_subtype"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        counts_list.append(X_sample.sum(axis=0))
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample, condition_col
        ].iloc[0]
        meta_list.append({sample_col: sample, condition_col: condition})
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    return counts_df, meta_df

print("Function defined")

Function defined


In [7]:
# Cell 4 — Loop DE across T cell subtypes (TUMOR vs BLOOD)
# Exclude B cell contamination cluster
subtypes_de = [
    "Resting T cells",
    "Naive/Memory T cells",
    "Activated T cells",
    "NK/Cytotoxic T cells"
]

all_results_tcell = {}

for ct in subtypes_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata1_tcells_raw,
            cell_type=ct,
            sample_col="patient",
            condition_col="tissue"
        )

        # Filter to TUMOR vs BLOOD only
        mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
        counts_df = counts_df[mask]
        meta_df = meta_df[mask]

        condition_counts = meta_df["tissue"].value_counts()
        print(f"  Samples per condition: {condition_counts.to_dict()}")
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples")
            continue

        inference = DefaultInference()
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~tissue",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["tissue", "TUMOR", "BLOOD"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df

        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")

        print(f"  Significant DEGs: {len(sig_df)}")
        if len(sig_df) > 0:
            print("\n  Top upregulated in TUMOR:")
            print(sig_df[sig_df["log2FoldChange"] > 0].head(5)[["log2FoldChange", "padj"]].to_string())
            print("\n  Top downregulated in TUMOR:")
            print(sig_df[sig_df["log2FoldChange"] < 0].head(5)[["log2FoldChange", "padj"]].to_string())

        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(RESULTS_DIR / f"GSE114725_DE_tcell_subcluster_{ct_clean}_tumor_vs_blood.csv")
        sig_df.to_csv(RESULTS_DIR / f"GSE114725_DE_tcell_subcluster_{ct_clean}_tumor_vs_blood_significant.csv")

        all_results_tcell[ct] = {"full": results_df, "sig": sig_df}

    except Exception as e:
        print(f"  Failed: {e}")

print("\nT cell subcluster DE complete")


Running DE for: Resting T cells

Resting T cells: 5336 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}
  Samples per condition: {'TUMOR': 3, 'BLOOD': 2}


Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 12.00 seconds.

Fitting dispersion trend curve...
... done in 1.60 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 9.91 seconds.

Fitting LFCs...
... done in 8.74 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 4.28 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     41.165745       -0.811676  0.614984 -1.319831  0.186891  0.833955
A2M      10.344423        2.581417  1.049991  2.458514  0.013951  0.236685
A4GALT    0.199333       -0.332385  4.672555 -0.071136  0.943290       NaN
AAAS     13.591379       -0.197883  0.813988 -0.243103  0.807926  0.999072
AACS      7.073269       -1.599871  1.247929 -1.282021  0.199835       NaN
...            ...             ...       ...       ...       ...       ...
ZXDC     16.575442       -0.166120  1.245672 -0.133358  0.893910  0.999072
ZYG11B   20.307977       -0.130109  0.856191 -0.151963  0.879216  0.999072
ZYX     140.431555       -0.073179  0.447791 -0.163422  0.870186  0.999072
ZZEF1    41.327815       -0.226545  0.514045 -0.440709  0.659424  0.999072
ZZZ3     16.729293       -0.128653  0.787460 -0.163377  0.870221  0.999072

[14800 rows x 6 columns]
  Significant 

...

Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     72.777015       -0.979360  0.555366 -1.763450  0.077825  0.632494
A2M      78.567217        0.175798  0.660950  0.265978  0.790256  0.996443
A4GALT    2.723012       -3.694022  2.537176 -1.455958  0.145404       NaN
AAAS     34.007348        0.121492  0.620172  0.195900  0.844688  0.996443
AACS     19.784903        0.303977  0.828265  0.367004  0.713616  0.996443
...            ...             ...       ...       ...       ...       ...
ZXDC     29.807146       -0.390338  0.701428 -0.556490  0.577876  0.996443
ZYG11B   41.590558       -0.045022  0.612675 -0.073485  0.941420  0.998151
ZYX     375.996482        0.174563  0.319574  0.546238  0.584902  0.996443
ZZEF1    85.340620       -0.038967  0.434004 -0.089785  0.928458  0.997225
ZZZ3     31.800876       -1.193302  0.688456 -1.733302  0.083042  0.654887

[14800 rows x 6 columns]
  Significant 

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 10.72 seconds.

Fitting dispersion trend curve...
... done in 1.46 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 9.93 seconds.

Fitting LFCs...
... done in 7.81 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.94 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     42.099833       -0.778679  0.643330 -1.210388  0.226130  0.947189
A2M      23.736901        3.041933  1.005647  3.024853  0.002488  0.084541
A4GALT    0.033626        1.305883  4.894022  0.266832  0.789598       NaN
AAAS     26.835264        0.523205  0.675459  0.774593  0.438580  0.999274
AACS     13.059452        0.261778  0.913194  0.286662  0.774371  0.999274
...            ...             ...       ...       ...       ...       ...
ZXDC     24.451051        0.387615  0.686939  0.564265  0.572574  0.999274
ZYG11B   29.528346        0.269995  0.630975  0.427901  0.668723  0.999274
ZYX     270.930596        0.077554  0.316753  0.244840  0.806580  0.999274
ZZEF1    63.395578        0.140197  0.426525  0.328697  0.742385  0.999274
ZZZ3     22.575153       -0.279686  0.705997 -0.396158  0.691989  0.999274

[14800 rows x 6 columns]
  Significant 

In [8]:
# Cell 5 — Pathway enrichment on T cell subcluster DE results
print("Running pathway enrichment on T cell subcluster DE results...")

for ct, res in all_results_tcell.items():
    print(f"\n{'='*40}")
    print(f"Pathways: {ct}")

    full_df = res["full"].dropna(subset=["padj"])
    sig_df = res["sig"]

    background = full_df.index.tolist()
    up_genes = sig_df[sig_df["log2FoldChange"] > 0].index.tolist()
    down_genes = sig_df[sig_df["log2FoldChange"] < 0].index.tolist()

    print(f"  Up: {len(up_genes)}, Down: {len(down_genes)}, Background: {len(background)}")

    for direction, gene_list in [("up", up_genes), ("down", down_genes)]:
        if len(gene_list) < 10:
            print(f"  Skipping {direction} - too few genes ({len(gene_list)})")
            continue
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
                background=background,
                outdir=None,
                verbose=False
            )
            sig_paths = enr.results[enr.results["Adjusted P-value"] < 0.05].copy()
            print(f"  {direction}: {len(sig_paths)} significant pathways")
            if len(sig_paths) > 0:
                print(sig_paths[["Gene_set", "Term", "Adjusted P-value"]].head(5).to_string())

            ct_clean = ct.replace("/", "_").replace(" ", "_")
            enr.results.to_csv(
                RESULTS_DIR / f"GSE114725_pathways_tcell_subcluster_{ct_clean}_{direction}.csv",
                index=False
            )
        except Exception as e:
            print(f"  {direction} failed: {e}")

print("\nPathway enrichment complete")

Running pathway enrichment on T cell subcluster DE results...

Pathways: Resting T cells
  Up: 157, Down: 121, Background: 8245
  up: 88 significant pathways
               Gene_set                           Term  Adjusted P-value
0  MSigDB_Hallmark_2020  TNF-alpha Signaling via NF-kB      5.844030e-39
1  MSigDB_Hallmark_2020          Inflammatory Response      2.111333e-11
2  MSigDB_Hallmark_2020      Interferon Gamma Response      1.589954e-10
3  MSigDB_Hallmark_2020                        Hypoxia      1.589954e-10
4  MSigDB_Hallmark_2020                     Complement      1.077376e-09
  down: 4 significant pathways
                Gene_set                                             Term  Adjusted P-value
0   MSigDB_Hallmark_2020                                   Myc Targets V1      8.948399e-06
18       KEGG_2021_Human                                         Ribosome      9.908104e-95
19       KEGG_2021_Human                              Coronavirus disease      4.544806e-90
20   

In [12]:
# Cell 6 updated — Heatmap with significance masking
fig, ax = plt.subplots(figsize=(10, 14), facecolor="white")

# Build significance mask
sig_matrix = pd.DataFrame(False, index=all_genes, columns=list(subtypes.keys()))

for label, filename in subtypes.items():
    sig_df = pd.read_csv(
        RESULTS_DIR / f"GSE114725_DE_tcell_subcluster_{filename}_tumor_vs_blood_significant.csv",
        index_col=0
    )
    for gene in all_genes:
        if gene in sig_df.index:
            sig_matrix.loc[gene, label] = True

# Plot base heatmap in grey for non-significant
grey_matrix = np.where(sig_matrix.values, np.nan, lfc_matrix.values)
sig_lfc = np.where(sig_matrix.values, lfc_matrix.values, np.nan)

vmax = np.nanmax(np.abs(lfc_matrix.values))

# Grey background for non-significant
ax.imshow(
    np.ones_like(lfc_matrix.values),
    cmap="Greys",
    aspect="auto",
    vmin=0,
    vmax=1,
    alpha=0.2
)

# Coloured overlay for significant
im = ax.imshow(
    sig_lfc,
    cmap="RdBu_r",
    aspect="auto",
    vmin=-vmax,
    vmax=vmax
)

ax.set_xticks(range(len(subtypes)))
ax.set_xticklabels(list(subtypes.keys()), rotation=30, ha="right", fontsize=11)
ax.set_yticks(range(len(all_genes)))
ax.set_yticklabels(all_genes, fontsize=9)

ax.axhline(y=len(all_up) - 0.5, color="black", linewidth=1.5, linestyle="--")

ax.text(-0.5, len(all_up)/2 - 0.5, "UP in\ntumour",
        ha="right", va="center", fontsize=10, color="red", fontweight="bold")
ax.text(-0.5, len(all_up) + len(all_down)/2 - 0.5, "DOWN in\ntumour",
        ha="right", va="center", fontsize=10, color="blue", fontweight="bold")

# Add note about grey cells
ax.text(0.5, -0.02, "Grey = not significant (padj > 0.05 or |LFC| < 0.5)",
        transform=ax.transAxes, ha="center", fontsize=9,
        color="grey", style="italic")

plt.colorbar(im, ax=ax, label="Log2 Fold Change (Tumour vs Blood)", shrink=0.4)
ax.set_title("T Cell Subcluster DE — Tumour vs Blood\nLog2 Fold Change per Subtype (grey = not significant)",
             fontsize=13, pad=15)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "GSE114725_tcell_subcluster_DE_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.close()
print("Saved")

Saved


In [3]:
# Cell 3 — Pseudobulk aggregation function
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        counts_list.append(X_sample.sum(axis=0))
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample, condition_col
        ].iloc[0]
        meta_list.append({sample_col: sample, condition_col: condition})
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    return counts_df, meta_df

print("Function defined")

Function defined


In [5]:
# Cell 15 — Pathway enrichment on GSE176078 DE results
de_files2 = list(RESULTS_DIR.glob("GSE176078_DE_*_significant.csv"))
print(f"Found {len(de_files2)} significant result files")

for sig_path in sorted(de_files2):
    # Get corresponding full results file
    full_path = Path(str(sig_path).replace("_significant", ""))
    if not full_path.exists():
        print(f"Full results not found for {sig_path.name}")
        continue

    label = sig_path.stem.replace("GSE176078_DE_", "").replace("_significant", "")
    print(f"\n{label}")

    full_df = pd.read_csv(full_path, index_col=0).dropna(subset=["padj"])
    sig_df = pd.read_csv(sig_path, index_col=0)

    background = full_df.index.tolist()
    up_genes = sig_df[sig_df["log2FoldChange"] > 0].index.tolist()
    down_genes = sig_df[sig_df["log2FoldChange"] < 0].index.tolist()

    print(f"  Up: {len(up_genes)}, Down: {len(down_genes)}, Background: {len(background)}")

    for direction, gene_list in [("up", up_genes), ("down", down_genes)]:
        if len(gene_list) < 10:
            continue
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
                background=background,
                outdir=None,
                verbose=False
            )
            sig_paths = enr.results[enr.results["Adjusted P-value"] < 0.05].copy()
            print(f"  {direction}: {len(sig_paths)} significant pathways")
            if len(sig_paths) > 0:
                print(sig_paths[["Gene_set", "Term", "Adjusted P-value"]].head(3).to_string())
            enr.results.to_csv(
                RESULTS_DIR / f"GSE176078_pathways_{label}_{direction}.csv",
                index=False
            )
        except Exception as e:
            print(f"  {direction} failed: {e}")

print("\nPathway enrichment complete")

Found 18 significant result files

B_cells_HER2+_vs_ER+
  Up: 0, Down: 0, Background: 17736

B_cells_TNBC_vs_ER+
  Up: 0, Down: 1, Background: 17736

B_cells_TNBC_vs_HER2+
  Up: 0, Down: 0, Background: 17736

CD8_T_cells_HER2+_vs_ER+
  Up: 0, Down: 0, Background: 20026

CD8_T_cells_TNBC_vs_ER+
  Up: 3, Down: 2, Background: 20026

CD8_T_cells_TNBC_vs_HER2+
  Up: 0, Down: 0, Background: 20026

Macrophages_HER2+_vs_ER+
  Up: 0, Down: 0, Background: 20331

Macrophages_TNBC_vs_ER+
  Up: 1, Down: 1, Background: 20331

Macrophages_TNBC_vs_HER2+
  Up: 0, Down: 0, Background: 20331

Naive_memory_T_cells_HER2+_vs_ER+
  Up: 0, Down: 0, Background: 20447

Naive_memory_T_cells_TNBC_vs_ER+
  Up: 0, Down: 0, Background: 20447

Naive_memory_T_cells_TNBC_vs_HER2+
  Up: 0, Down: 0, Background: 20447

NK_cells_HER2+_vs_ER+
  Up: 0, Down: 0, Background: 17512

NK_cells_TNBC_vs_ER+
  Up: 3, Down: 1, Background: 17512

NK_cells_TNBC_vs_HER2+
  Up: 0, Down: 0, Background: 17512

T_cells_HER2+_vs_ER+
  Up: 0,

In [13]:
pip install sccoda

Note: you may need to restart the kernel to use updated packages.Collecting sccoda
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 524.3 kB/s eta 0:00:03
   ------------------ --------------------- 0.8/1.7 MB 671.3 kB/s eta 0:00:02
   ------------------------- -------------- 1.0/1.7 MB 719.5 kB/s eta 0:00:01
   ------------------------- -------------- 1.0/1.7 MB 719.5 kB/s eta 0:

  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------ --------- 25/33 [rich]
   ------------------------------- -------- 26/33 [xarray-einstats]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------------------- ------- 27/33 [rpy2-robjects]
   -------------------

In [14]:
# Test import
import sccoda
print(sccoda.__version__)

0.1.9


In [16]:
import sccoda
from sccoda.util import comp_ana as ca
from sccoda.util import cell_composition_data as dat
print("scCODA imported successfully")



Failed to import TF-Keras. Please note that TF-Keras is not installed by default when you install TensorFlow Probability. This is so that JAX-only users do not have to install TensorFlow or TF-Keras. To use TensorFlow Probability with TensorFlow, please install the tf-keras or tf-keras-nightly package.
This can be be done through installing the tensorflow-probability[tf] extra.




ModuleNotFoundError: No module named 'tf_keras'

In [15]:
# Cell — scCODA composition analysis (GSE176078 subtypes)
import sccoda
from sccoda.util import comp_ana as ca
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"

# Load proportion data
props = pd.read_csv(
    PROJECT_DIR / "results" / "phase2_clustering_v2" / 
    "GSE176078_celltype_proportions_by_subtype.csv",
    index_col=0
)

print(props.head())
print(props.shape)



Failed to import TF-Keras. Please note that TF-Keras is not installed by default when you install TensorFlow Probability. This is so that JAX-only users do not have to install TensorFlow or TF-Keras. To use TensorFlow Probability with TensorFlow, please install the tf-keras or tf-keras-nightly package.
This can be be done through installing the tensorflow-probability[tf] extra.




ModuleNotFoundError: No module named 'tf_keras'